# RPG and SASRec hyperparameter analysis

Reproduction of Figure 6, the inference candidate-budget behaviour, and the seeded training-grid sweeps for RPG (learning rate and temperature) and SASRec (learning rate, dropout, self-attention blocks).

Sources collected by `scripts/collect_grid_results.py`: `fig6_grid.csv`, `infer_grid.csv`, `train_grid.csv`, `sasrec_train_grid.csv`. Decode grids report mean and standard deviation over evaluation seeds; training grids report mean and standard deviation over training seeds. Sweep optima are selected on validation (RPG val NDCG@10, SASRec val NDCG@20) and reported on test. The notebook is safe to run on partial data: panels with no rows yet are skipped.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

BASE = Path.cwd()
ROOT = BASE if (BASE / "scripts").exists() else BASE.parent
SEARCH = [ROOT, ROOT / "results"]
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

WONG = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})

DATASETS = ["sports_and_outdoors", "beauty", "toys_and_games", "cds_and_vinyl"]
LABELS = {"sports_and_outdoors": "Sports", "beauty": "Beauty", "toys_and_games": "Toys", "cds_and_vinyl": "CDs"}
TRAIN_DATASETS = ["sports_and_outdoors", "beauty", "toys_and_games"]
TUNED = {"sports_and_outdoors": (0.003, 0.03), "beauty": (0.01, 0.03), "toys_and_games": (0.003, 0.03), "cds_and_vinyl": (0.001, 0.03)}
SASREC_RELEASED = {"lr": 0.001, "dropout": 0.5, "n_blocks": 2}
BASE_KQ = {"sports_and_outdoors": (30, 5), "beauty": (200, 3), "toys_and_games": (20, 3), "cds_and_vinyl": (500, 5)}
PAPER_BEST_NDCG10 = {"sports_and_outdoors": 0.0263, "beauty": 0.0464, "toys_and_games": 0.0490, "cds_and_vinyl": 0.0415}

In [ ]:
def find_csv(name):
    for d in SEARCH:
        p = d / name
        if p.exists():
            return p
    return None


def load_decode(name):
    cols = ["dataset", "num_beams", "n_edges", "propagation_steps", "metric", "mean", "std", "n_seeds"]
    p = find_csv(name)
    return pd.read_csv(p) if p is not None else pd.DataFrame(columns=cols)


def load_train(name):
    cols = ["dataset", "lr", "temperature", "metric", "mean", "std", "n_seeds", "val_ndcg10_mean", "val_ndcg10_std"]
    p = find_csv(name)
    if p is None:
        return pd.DataFrame(columns=cols)
    df = pd.read_csv(p)
    if "mean" not in df.columns and "value" in df.columns:
        df = df.rename(columns={"value": "mean"})
        df["std"] = 0.0
        df["n_seeds"] = 1
    if "val_ndcg10_mean" not in df.columns:
        df["val_ndcg10_mean"] = np.nan
    return df


def load_sasrec(name):
    cols = ["dataset", "lr", "dropout", "n_blocks", "metric", "test_mean", "test_std", "n_seeds", "val_ndcg20_mean", "val_ndcg20_std"]
    p = find_csv(name)
    return pd.read_csv(p) if p is not None else pd.DataFrame(columns=cols)


fig6 = load_decode("fig6_grid.csv")
infer = load_decode("infer_grid.csv")
train = load_train("train_grid.csv")
sasrec = load_sasrec("sasrec_train_grid.csv")
pd.Series({"fig6_rows": len(fig6), "infer_rows": len(infer), "train_rows": len(train), "sasrec_rows": len(sasrec)})

## 1. Figure 6 replication

Decode-only sweep of beam size `b`, edges per node `k`, and iteration steps `q`, one factor at a time around the paper base `b=10, k=50, q=2`, evaluated at NDCG@10. The dashed line marks the paper best-m NDCG@10 from Table 2. Sports is the strict replication; the other datasets extend Figure 6 beyond the original paper.

In [ ]:
def fig6_panels(df, dataset, metric="ndcg@10", base=(10, 50, 2)):
    sub = df[(df["dataset"] == dataset) & (df["metric"] == metric)]
    if sub.empty:
        print("no fig6 data for", dataset)
        return
    b0, k0, q0 = base
    beam = sub[(sub["n_edges"] == k0) & (sub["propagation_steps"] == q0)].sort_values("num_beams")
    edge = sub[(sub["num_beams"] == b0) & (sub["propagation_steps"] == q0)].sort_values("n_edges")
    step = sub[(sub["num_beams"] == b0) & (sub["n_edges"] == k0)].sort_values("propagation_steps")
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
    for ax, (x, d, title) in zip(axes, [("num_beams", beam, "beam size b"), ("n_edges", edge, "edges per node k"), ("propagation_steps", step, "iteration steps q")]):
        if not d.empty:
            ax.errorbar(d[x], d["mean"], yerr=d["std"], marker="o", color=WONG[0], capsize=3)
        ax.axhline(PAPER_BEST_NDCG10.get(dataset), ls="--", color=WONG[6], lw=1, label="paper best-m")
        ax.set_xlabel(title)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0].set_ylabel(metric.upper())
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle("Figure 6 replication: " + LABELS[dataset])
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"fig6_{dataset}.{ext}", bbox_inches="tight")
    plt.show()


fig6_panels(fig6, "sports_and_outdoors")

In [ ]:
for ds in ["beauty", "toys_and_games", "cds_and_vinyl"]:
    fig6_panels(fig6, ds)

## 2. Inference candidate budget

Graph-constrained decoding returns at most `num_beams` candidates, so NDCG@k is undefined for `k` greater than `num_beams`. The first panel set sweeps beam size at each dataset base and overlays NDCG@{5,10,50,100}: the @50 and @100 curves only appear once beams cross 50 and 100. The edge and step sweeps are held at `num_beams=200` so all cutoffs stay valid.

In [ ]:
CUTOFF_METRICS = ["ndcg@5", "ndcg@10", "ndcg@50", "ndcg@100"]


def candidate_cap(df):
    present = [d for d in DATASETS if not df[df["dataset"] == d].empty]
    if not present:
        print("no inference data")
        return
    fig, axes = plt.subplots(1, len(present), figsize=(3.4 * len(present), 3.4))
    axes = np.atleast_1d(axes)
    for ax, ds in zip(axes, present):
        k0, q0 = BASE_KQ[ds]
        sub = df[(df["dataset"] == ds) & (df["n_edges"] == k0) & (df["propagation_steps"] == q0)]
        for i, m in enumerate(CUTOFF_METRICS):
            line = sub[sub["metric"] == m].sort_values("num_beams")
            if not line.empty:
                ax.errorbar(line["num_beams"], line["mean"], yerr=line["std"], marker="o", capsize=2, color=WONG[i], label=m.upper())
        ax.set_xscale("log")
        ax.set_title(LABELS[ds])
        ax.set_xlabel("beam size b")
    axes[0].set_ylabel("NDCG")
    axes[0].legend(frameon=False, fontsize=8)
    fig.suptitle("Candidate budget: NDCG@k is undefined below k beams")
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"candidate_cap.{ext}", bbox_inches="tight")
    plt.show()


candidate_cap(infer)

In [ ]:
def fixed_beam_sweep(df, sweep_col, base_index, xlabel, fname, metric="ndcg@10", beams=200):
    present = [d for d in DATASETS if not df[df["dataset"] == d].empty]
    if not present:
        print("no inference data")
        return
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    for i, ds in enumerate(present):
        held = BASE_KQ[ds][base_index]
        hold_col = "propagation_steps" if sweep_col == "n_edges" else "n_edges"
        sub = df[(df["dataset"] == ds) & (df["metric"] == metric) & (df["num_beams"] == beams) & (df[hold_col] == held)].sort_values(sweep_col)
        if not sub.empty:
            ax.errorbar(sub[sweep_col], sub["mean"], yerr=sub["std"], marker="o", capsize=2, color=WONG[i], label=LABELS[ds])
    ax.set_xlabel(xlabel)
    ax.set_ylabel(metric.upper())
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{fname}.{ext}", bbox_inches="tight")
    plt.show()


fixed_beam_sweep(infer, "n_edges", 1, "edges per node k", "edge_sweep")
fixed_beam_sweep(infer, "propagation_steps", 0, "iteration steps q", "step_sweep")

## 3. RPG training grid: learning rate and temperature

Full retrain over learning rate and temperature at each dataset best-m, three training seeds per cell. Heatmap cells show the seed mean test NDCG@10 with the seed standard deviation below it. The orange box marks the released tuned cell; the green box marks the configuration selected on validation NDCG@10. Selecting on validation, not test, avoids tuning on the evaluation set.

In [ ]:
def best_by_val_rpg(df, dataset, metric="ndcg@10"):
    sub = df[(df["dataset"] == dataset) & (df["metric"] == metric)].copy()
    sub["val_ndcg10_mean"] = pd.to_numeric(sub["val_ndcg10_mean"], errors="coerce")
    sub = sub[sub["val_ndcg10_mean"].notna()]
    if sub.empty:
        return None
    return sub.loc[sub["val_ndcg10_mean"].idxmax()]


def lr_temp_heatmap(df, dataset, metric="ndcg@10"):
    sub = df[(df["dataset"] == dataset) & (df["metric"] == metric)]
    if sub.empty:
        print("no training data for", dataset)
        return
    grid = sub.pivot_table(index="lr", columns="temperature", values="mean")
    err = sub.pivot_table(index="lr", columns="temperature", values="std")
    fig, ax = plt.subplots(figsize=(4.4, 3.8))
    im = ax.imshow(grid.values, origin="lower", aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(grid.columns)))
    ax.set_xticklabels(grid.columns)
    ax.set_yticks(range(len(grid.index)))
    ax.set_yticklabels(grid.index)
    ax.set_xlabel("temperature")
    ax.set_ylabel("learning rate")
    ax.grid(False)
    for yi in range(grid.shape[0]):
        for xi in range(grid.shape[1]):
            v = grid.values[yi, xi]
            e = err.values[yi, xi]
            if np.isfinite(v):
                ax.text(xi, yi, f"{v:.4f}\n{e:.4f}", ha="center", va="center", color="white", fontsize=7)
    cols, idx = list(grid.columns), list(grid.index)
    lr_t, te_t = TUNED[dataset]
    if lr_t in idx and te_t in cols:
        ax.add_patch(plt.Rectangle((cols.index(te_t) - 0.5, idx.index(lr_t) - 0.5), 1, 1, fill=False, edgecolor=WONG[1], lw=2.2))
    best = best_by_val_rpg(df, dataset, metric)
    if best is not None and best["lr"] in idx and best["temperature"] in cols:
        ax.add_patch(plt.Rectangle((cols.index(best["temperature"]) - 0.5, idx.index(best["lr"]) - 0.5), 1, 1, fill=False, edgecolor=WONG[2], lw=2.2))
        ax.set_title(f"{LABELS[dataset]} {metric.upper()}: val-selected {best['mean']:.4f} at lr={best['lr']}, t={best['temperature']}")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"lrtemp_{dataset}.{ext}", bbox_inches="tight")
    plt.show()


for ds in TRAIN_DATASETS:
    lr_temp_heatmap(train, ds)

In [ ]:
def rpg_summary(df, metric="ndcg@10"):
    rows = []
    for ds in TRAIN_DATASETS:
        sub = df[(df["dataset"] == ds) & (df["metric"] == metric)]
        if sub.empty:
            continue
        best = best_by_val_rpg(df, ds, metric)
        lr_t, te_t = TUNED[ds]
        tuned = sub[(sub["lr"] == lr_t) & (sub["temperature"] == te_t)]
        tm = float(tuned["mean"].iloc[0]) if not tuned.empty else np.nan
        ts = float(tuned["std"].iloc[0]) if not tuned.empty else np.nan
        rows.append({
            "dataset": LABELS[ds],
            "sel_lr": None if best is None else best["lr"],
            "sel_temperature": None if best is None else best["temperature"],
            "sel_test_mean": np.nan if best is None else round(float(best["mean"]), 5),
            "sel_test_std": np.nan if best is None else round(float(best["std"]), 5),
            "tuned_test_mean": round(tm, 5),
            "tuned_test_std": round(ts, 5),
            "delta": np.nan if best is None else round(float(best["mean"]) - tm, 5),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(TAB_DIR / "train_grid_best.csv", index=False)
    return out


rpg_summary(train)

## 4. Seed robustness

Pre-registered decision rule, fixed before inspecting all seeds: the validation-selected configuration counts as a genuine improvement over the released tuned cell only if its test lower bound (mean minus standard deviation) exceeds the tuned cell test upper bound (mean plus standard deviation). Overlapping intervals are reported as seed noise.

In [ ]:
def seed_robustness(df, metric="ndcg@10"):
    rows = []
    for ds in TRAIN_DATASETS:
        sub = df[(df["dataset"] == ds) & (df["metric"] == metric)]
        if sub.empty:
            continue
        best = best_by_val_rpg(df, ds, metric)
        lr_t, te_t = TUNED[ds]
        tuned = sub[(sub["lr"] == lr_t) & (sub["temperature"] == te_t)]
        if best is None or tuned.empty:
            continue
        tm, ts = float(tuned["mean"].iloc[0]), float(tuned["std"].iloc[0])
        bm, bs = float(best["mean"]), float(best["std"])
        rows.append({
            "dataset": LABELS[ds],
            "best_minus_tuned": round(bm - tm, 5),
            "best_lower": round(bm - bs, 5),
            "tuned_upper": round(tm + ts, 5),
            "non_overlapping": bool((bm - bs) > (tm + ts)),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(TAB_DIR / "seed_robustness.csv", index=False)
    return out


seed_robustness(train)

## 5. SASRec sweep: learning rate, dropout, self-attention blocks

Full retrain over the tunable SASRec hyperparameters (learning rate, dropout, number of self-attention blocks), three training seeds per cell, selected on validation NDCG@20. The heatmap fixes blocks at the released `b=2` and sweeps learning rate against dropout; the block panel fixes the released learning rate and dropout and sweeps `b`. The orange box marks the released configuration; the green box marks the validation-selected cell.

In [ ]:
def best_by_val_sasrec(df, dataset, metric="ndcg@10"):
    sub = df[(df["dataset"] == dataset) & (df["metric"] == metric)].copy()
    sub["val_ndcg20_mean"] = pd.to_numeric(sub["val_ndcg20_mean"], errors="coerce")
    sub = sub[sub["val_ndcg20_mean"].notna()]
    if sub.empty:
        return None
    return sub.loc[sub["val_ndcg20_mean"].idxmax()]


def sasrec_heatmap(df, dataset, metric="ndcg@10", blocks=2):
    sub = df[(df["dataset"] == dataset) & (df["metric"] == metric) & (df["n_blocks"] == blocks)]
    if sub.empty:
        print("no sasrec data for", dataset)
        return
    grid = sub.pivot_table(index="lr", columns="dropout", values="test_mean")
    err = sub.pivot_table(index="lr", columns="dropout", values="test_std")
    fig, ax = plt.subplots(figsize=(4.4, 3.8))
    im = ax.imshow(grid.values, origin="lower", aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(grid.columns)))
    ax.set_xticklabels(grid.columns)
    ax.set_yticks(range(len(grid.index)))
    ax.set_yticklabels(grid.index)
    ax.set_xlabel("dropout")
    ax.set_ylabel("learning rate")
    ax.grid(False)
    for yi in range(grid.shape[0]):
        for xi in range(grid.shape[1]):
            v = grid.values[yi, xi]
            e = err.values[yi, xi]
            if np.isfinite(v):
                ax.text(xi, yi, f"{v:.4f}\n{e:.4f}", ha="center", va="center", color="white", fontsize=7)
    cols, idx = list(grid.columns), list(grid.index)
    if SASREC_RELEASED["lr"] in idx and SASREC_RELEASED["dropout"] in cols:
        ax.add_patch(plt.Rectangle((cols.index(SASREC_RELEASED["dropout"]) - 0.5, idx.index(SASREC_RELEASED["lr"]) - 0.5), 1, 1, fill=False, edgecolor=WONG[1], lw=2.2))
    best = best_by_val_sasrec(df, dataset, metric)
    if best is not None and best["n_blocks"] == blocks and best["lr"] in idx and best["dropout"] in cols:
        ax.add_patch(plt.Rectangle((cols.index(best["dropout"]) - 0.5, idx.index(best["lr"]) - 0.5), 1, 1, fill=False, edgecolor=WONG[2], lw=2.2))
    ax.set_title(f"SASRec {LABELS[dataset]} {metric.upper()} (b={blocks})")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"sasrec_lrdrop_{dataset}.{ext}", bbox_inches="tight")
    plt.show()


for ds in TRAIN_DATASETS:
    sasrec_heatmap(sasrec, ds)

In [ ]:
def sasrec_block_sweep(df, metric="ndcg@10"):
    present = [d for d in TRAIN_DATASETS if not df[df["dataset"] == d].empty]
    if not present:
        print("no sasrec data")
        return
    fig, ax = plt.subplots(figsize=(5.2, 3.4))
    for i, ds in enumerate(present):
        sub = df[(df["dataset"] == ds) & (df["metric"] == metric) & (df["lr"] == SASREC_RELEASED["lr"]) & (df["dropout"] == SASREC_RELEASED["dropout"])].sort_values("n_blocks")
        if not sub.empty:
            ax.errorbar(sub["n_blocks"], sub["test_mean"], yerr=sub["test_std"], marker="o", capsize=3, color=WONG[i], label=LABELS[ds])
    ax.set_xlabel("self-attention blocks b")
    ax.set_ylabel(metric.upper())
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"sasrec_blocks.{ext}", bbox_inches="tight")
    plt.show()


sasrec_block_sweep(sasrec)

In [ ]:
def sasrec_summary(df, metric="ndcg@10"):
    rows = []
    for ds in TRAIN_DATASETS:
        sub = df[(df["dataset"] == ds) & (df["metric"] == metric)]
        if sub.empty:
            continue
        best = best_by_val_sasrec(df, ds, metric)
        rel = sub[(sub["lr"] == SASREC_RELEASED["lr"]) & (sub["dropout"] == SASREC_RELEASED["dropout"]) & (sub["n_blocks"] == SASREC_RELEASED["n_blocks"])]
        rm = float(rel["test_mean"].iloc[0]) if not rel.empty else np.nan
        rs = float(rel["test_std"].iloc[0]) if not rel.empty else np.nan
        rows.append({
            "dataset": LABELS[ds],
            "sel_lr": None if best is None else best["lr"],
            "sel_dropout": None if best is None else best["dropout"],
            "sel_blocks": None if best is None else int(best["n_blocks"]),
            "sel_test_mean": np.nan if best is None else round(float(best["test_mean"]), 5),
            "sel_test_std": np.nan if best is None else round(float(best["test_std"]), 5),
            "released_test_mean": round(rm, 5),
            "released_test_std": round(rs, 5),
            "delta": np.nan if best is None else round(float(best["test_mean"]) - rm, 5),
        })
    out = pd.DataFrame(rows)
    if not out.empty:
        out.to_csv(TAB_DIR / "sasrec_grid_best.csv", index=False)
    return out


sasrec_summary(sasrec)